# Push 80 Valid Colab
Notebook ini untuk mendorong macro-F1 ke 0.78-0.80 secara valid (tanpa leakage), dengan trial model kuat + evaluasi test per trial + class multiplier tuning.

## 0) Setup Runtime
Pilih GPU T4: Runtime > Change runtime type > T4 GPU.

In [ ]:
import os
HF_TOKEN = 'hf_xxx_your_token_here'  # placeholder
os.environ['HF_TOKEN'] = HF_TOKEN
!git clone https://github.com/kzquandary/mbg-sentiment.git
%cd mbg-sentiment
!git checkout main
!python3 -m pip install -r requirements.txt
!nvidia-smi


## 1) Pilih Dataset
Default `v3_aug` (augment minority).

In [ ]:
# Toggle dataset
DATASET_PATH = 'data/dataset_relabel_mbg_improved_v3_aug.csv'  # default
# DATASET_PATH = 'data/dataset_relabel_mbg_improved_v2_boost.csv'
# DATASET_PATH = 'data/dataset_relabel_mbg_improved.csv'
assert os.path.exists(DATASET_PATH), f'Dataset tidak ditemukan: {DATASET_PATH}'
print('Dataset:', DATASET_PATH)
print('Config:', 'src/resources/step7_final_production.json')


## 2) Split + Preprocess + Internal Val + Balance (CPU)

In [ ]:
!python3 src/05_split_data.py --input "$DATASET_PATH" --text-col text --label-col Labeling_Sentimen --train-ratio 0.7 --val-ratio 0 --test-ratio 0.3
!python3 src/03_preprocess_text.py --train-input data/train.csv --test-input data/test.csv --text-col text


In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
df = pd.read_csv('data/train.csv')
train_sub, val_sub = train_test_split(df, test_size=0.15, random_state=42, stratify=df['Labeling_Sentimen'].astype(str))
train_sub.to_csv('data/train_sub.csv', index=False, encoding='utf-8-sig')
val_sub.to_csv('data/val_sub.csv', index=False, encoding='utf-8-sig')
print('train_sub', len(train_sub), train_sub['Labeling_Sentimen'].value_counts().to_dict())
print('val_sub  ', len(val_sub), val_sub['Labeling_Sentimen'].value_counts().to_dict())


In [ ]:
!python3 src/05b_balance_train.py --input data/train_sub.csv --label-col Labeling_Sentimen --target-by-label-json src/resources/train_balance_push80_valid.json --output data/train_sub_balanced.csv --log-output outputs/train_sub_balance_log.json


## 3) Train + Evaluate (Single Best Config, GPU)
Mode cepat: pakai 1 konfigurasi terbaik (`step7_final_production.json`), bukan 3 trial.

In [ ]:
import os, json
import pandas as pd
from IPython.display import display, Image

os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True,max_split_size_mb:64'

!python3 src/07_indobert_bilstm.py --train data/train_sub_balanced.csv --val data/val_sub.csv --trial-configs-json src/resources/step7_final_production.json --max-trials 1
!python3 src/09b_tune_class_multipliers.py --val data/val_sub.csv --model-path models/best_indobert_bilstm.pt --output outputs/best_class_multipliers.json --summary-output outputs/class_multiplier_tuning_summary.json
!python3 src/09_evaluate.py --test data/test.csv --model-path models/best_indobert_bilstm.pt --class-multiplier-json outputs/best_class_multipliers.json

metrics = json.load(open('outputs/final_metrics.json', encoding='utf-8'))
report = pd.read_csv('outputs/classification_report.csv')
cm = pd.read_csv('outputs/confusion_matrix.csv', index_col=0)
print('Final Metrics')
display(pd.DataFrame([metrics])[['accuracy','precision_macro','recall_macro','f1_macro']])
print('Classification Report')
display(report)
print('Confusion Matrix (table)')
display(cm)
if os.path.exists('outputs/figures/confusion_matrix.png'):
    print('Confusion Matrix (figure)')
    display(Image(filename='outputs/figures/confusion_matrix.png'))


## 4) Cek Run Log

In [ ]:
import json
step7 = json.load(open('outputs/step7_best_config.json', encoding='utf-8'))
print('Best Config (Step 7):')
print(step7.get('best_config', {}))
print('Best Val F1:', step7.get('best_val_f1_macro'))


## 5) Output
- `outputs/final_metrics.json`
- `outputs/classification_report.csv`
- `outputs/confusion_matrix.csv`
- `outputs/figures/confusion_matrix.png`
- `outputs/step7_best_config.json`
- `outputs/best_class_multipliers.json`
- `outputs/class_multiplier_tuning_summary.json`
